# Vax vs. No vote

### Load python tools

In [ ]:
%load_ext lab_black

In [ ]:
import pandas as pd
import geopandas as gpd
import altair as alt
import altair_latimes as lat
import jenkspy

In [ ]:
alt.themes.register("latimes", lat.theme)
alt.themes.enable("stiles")

In [ ]:
pd.options.display.max_columns = 100
pd.options.display.max_rows = 1000
alt.data_transformers.disable_max_rows()
pd.options.display.max_colwidth = None

### FIPS codes for counties

In [ ]:
fips = pd.read_html(
    "https://www.nrcs.usda.gov/wps/portal/nrcs/detail/ca/home/?cid=nrcs143_013697",
    attrs={"class": "data"},
)[0]

In [ ]:
ca_fips = fips[fips["State"] == "CA"].copy()

In [ ]:
ca_fips["FIPS"] = ca_fips["FIPS"].astype("str").str[1:]
ca_fips["Name"] = ca_fips["Name"].str.upper()

### Vax percentage

In [ ]:
vax_url = "https://raw.githubusercontent.com/datadesk/california-coronavirus-data/master/cdph-vaccination-county-totals.csv"

In [ ]:
vax_src = pd.read_csv(vax_url, dtype={"fips": str})

In [ ]:
vax_slim = vax_src[vax_src["date"] == vax_df["date"].max()][
    ["county", "fips", "fully_vaccinated_percent", "population"]
]

In [ ]:
vax_src.head()

### Yes/No vote

In [ ]:
recall_src = pd.read_json(
    "data/processed/gov_race_change_counties.json", dtype={"fips": str}
)

In [ ]:
recall_src.rename(columns={"d_pct_21": "no_percent"}, inplace=True)

In [ ]:
recall_slim = recall_src[["county", "fips", "no_percent", "leader_2021"]]

In [ ]:
recall_src.head()

---

### Merge

In [ ]:
merged = pd.merge(recall_slim, vax_slim, on=["fips", "county"])

In [ ]:
merged.corr(method="pearson")

---

In [ ]:
domain = ["R", "D"]
range_ = ["#d94f54", "#5789b8"]

alt.Chart(merged).mark_circle(size=100).encode(
    x=alt.X(
        "no_percent",
        scale=alt.Scale(domain=[10, 100]),
        axis=alt.Axis(tickCount=6, title="No percentage"),
    ),
    y=alt.Y(
        "fully_vaccinated_percent",
        scale=alt.Scale(domain=[0.2, 1]),
        axis=alt.Axis(tickCount=6, title="One dose percentage", format="%"),
    ),
    #     size="population",
    color=alt.Color("leader_2021", scale=alt.Scale(domain=domain, range=range_)),
    tooltip=["county", "population"],
).properties(width=1000, height=1000)

In [ ]:
merged["fully_vaccinated_percent"] = round(merged["fully_vaccinated_percent"] * 100, 2)

In [ ]:
merged.to_csv("data/processed/no_vax_corr.csv", index=False)